In [1]:
from dotenv import load_dotenv, find_dotenv
from pageindex import PageIndexClient

_ = load_dotenv(find_dotenv())

client = PageIndexClient(
    index="gpt-6-luna",
    chat="gpt-6.1-sol"
)

## Index a document

In [2]:
doc = client.submit_document("../docs/HBR_data_scientist_article.pdf")

In [3]:
print(doc)

{'doc_id': 'pi-fe1f8a3fae2648b19201bcdeba0418a2', 'name': 'HBR_data_scientist_article.pdf'}


In [4]:
doc_id = doc['doc_id']

In [6]:
response = None
for event in client.chat(
    "So what exactly is a data scientist and why is it the sexiest job of the 21st century?",
    doc_id=doc_id,
    stream=True,
    citations=True,
    protocol="responses"
):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] browse_documents({})

[Tool call] get_document({"doc_name":"HBR_data_scientist_article.pdf"})

[Tool call] get_page_content({"doc_name":"HBR_data_scientist_article.pdf","pages":"1-5"})

[Tool call] get_page_content({"doc_name":"HBR_data_scientist_article.pdf","pages":"6-7,12-13"})
A **data scientist turns messy data into discoveries—and those discoveries into useful business decisions or products.** The article describes the role as a hybrid of “data hacker, analyst, communicator, and trusted adviser.” It’s not just about analyzing numbers; it’s also about making data usable and explaining what the findings mean. <cite doc="HBR_data_scientist_article.pdf" page="4"/><cite doc="HBR_data_scientist_article.pdf" page="5"/>

In practice, that means:
- **Preparing the evidence:** finding promising data sources, combining them, and cleaning incomplete or unstructured data. <cite doc="HBR_data_scientist_article.pdf" page="4"/>
- **Thinking like a scientist:** asking insightful ques

In [10]:
response

{'id': 'resp_5220d6d0974640bc83e1158ae9a9d63f',
 'object': 'response',
 'created_at': 1794569271,
 'model': 'gpt-6.1-sol',
 'status': 'completed',
 'output': [{'arguments': '{}',
   'call_id': 'call_M6DpCSysxVMLFharit8U7t0B',
   'name': 'browse_documents',
   'type': 'function_call',
   'id': 'fc_026802f6c000cb2e006af6f439a75487d2881357ea1e979e5d',
   'status': 'completed'},
  {'arguments': '{"doc_name":"HBR_data_scientist_article.pdf"}',
   'call_id': 'call_tteL0ZfBisoiRCdsOtuBoPqC',
   'name': 'get_document',
   'type': 'function_call',
   'id': 'fc_026802f6c000cb2e006af6f43c96c087d2a69090774912d957',
   'status': 'completed'},
  {'arguments': '{"doc_name":"HBR_data_scientist_article.pdf","pages":"1-5"}',
   'call_id': 'call_DEOUnQythiufi8GoVJkgmfsp',
   'name': 'get_page_content',
   'type': 'function_call',
   'id': 'fc_026802f6c000cb2e006af6f43f698487d2a6b990db62438198',
   'status': 'completed'},
  {'arguments': '{"doc_name":"HBR_data_scientist_article.pdf","pages":"6-7,12-13"}',

In [16]:
answer = "".join(
    part["text"]
    for item in response["output"]
    if item.get("type") == "message"
    for part in item["content"]
    if "text" in part
)

In [17]:
from IPython.display import display, Markdown

display(Markdown(answer))

A **data scientist turns messy data into discoveries—and those discoveries into useful business decisions or products.** The article describes the role as a hybrid of “data hacker, analyst, communicator, and trusted adviser.” It’s not just about analyzing numbers; it’s also about making data usable and explaining what the findings mean. <cite doc="HBR_data_scientist_article.pdf" page="4"/><cite doc="HBR_data_scientist_article.pdf" page="5"/>

In practice, that means:
- **Preparing the evidence:** finding promising data sources, combining them, and cleaning incomplete or unstructured data. <cite doc="HBR_data_scientist_article.pdf" page="4"/>
- **Thinking like a scientist:** asking insightful questions, forming hypotheses, and testing them—with curiosity as a defining trait. <cite doc="HBR_data_scientist_article.pdf" page="6"/>
- **Making discoveries useful:** writing code, communicating findings visually and verbally, and advising people on products, processes, and decisions. <cite doc="HBR_data_scientist_article.pdf" page="5"/>

### Why “the sexiest job”?
In the article’s **2012 context**, the appeal comes from a powerful combination: **huge business opportunity, scarce talent, and visible impact.** Companies were collecting unprecedented volumes and varieties of information, but demand for people who could extract value from it had raced ahead of supply. The necessary blend of technical ability, curiosity, and communication was rare. <cite doc="HBR_data_scientist_article.pdf" page="3"/><cite doc="HBR_data_scientist_article.pdf" page="4"/><cite doc="HBR_data_scientist_article.pdf" page="5"/>

The standout example is LinkedIn’s **“People You May Know.”** Jonathan Goldman explored patterns in members’ profiles and connections, then tested personalized recommendations. The resulting feature achieved a click-through rate 30% higher than other prompts to visit more pages, generated millions of new page views, and significantly improved LinkedIn’s growth trajectory. That illustrates the attraction: a data scientist could do more than report on a business—they could help change its direction. <cite doc="HBR_data_scientist_article.pdf" page="2"/><cite doc="HBR_data_scientist_article.pdf" page="3"/>